# 言い換えたデータを足して少ない学習データを補う（データ拡張）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/12_classification/data_augmentation.ipynb)

NLP クックブック 12-16 ／ テキスト分類

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

ラベル付きデータが各クラス 10 件しかない。学習データを機械的に言い換えて水増しすれば分類の精度が上がるのか、同義語の置換 (EDA)、逆翻訳、LLM による言い換えを比べて確かめたい。

（T4 の GPU ランタイムで、全体で 10〜15 分程度かかる。）

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import io, random, tarfile, urllib.request
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

def load_livedoor():
    """livedoor ニュースコーパス（CC BY-ND 2.1 JP。改変禁止なので、言い換えたデータは手元の実験だけに使い、配布しない）"""
    url = "https://www.rondhuit.com/download/ldcc-20140209.tar.gz"
    with urllib.request.urlopen(url) as r:
        tar = tarfile.open(fileobj=io.BytesIO(r.read()), mode="r:gz")
    rows = []
    for m in tar.getmembers():
        parts = m.name.split("/")
        if m.isfile() and len(parts) == 3 and parts[2] != "LICENSE.txt":
            lines = tar.extractfile(m).read().decode("utf-8").split("\n")
            rows.append({"label": parts[1], "title": lines[2], "text": "\n".join(lines[3:])})
    return pd.DataFrame(rows)

df = load_livedoor()
pool, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=0)
# タイトルだけで 9 クラスに分類する。学習データの選び方（乱数の種）を 5 通り変えて、結果のばらつきも見る
SEEDS, N = range(5), 10
real3x = {s: pool.groupby("label").sample(3 * N, random_state=s) for s in SEEDS}   # 比較用: 本物のデータを 3 倍（各クラス 30 件）集めた場合
small = {s: d.groupby("label").head(N) for s, d in real3x.items()}                  # 各クラス 10 件（計 90 件）の学習データ
used = sorted(set().union(*[set(d.index) for d in small.values()]))                 # 5 通りのどれかで使う文 = 言い換える対象
print("学習データ:", len(small[0]), "件 × 5 通り / 言い換える文:", len(used), "件 / テスト:", len(test), "件")
augs = {}   # 手法名 → {文の番号: [言い換え 1, 言い換え 2]}

学習データ: 90 件 × 5 通り / 言い換える文: 428 件 / テスト: 1474 件


In [3]:
# (1) EDA: 同義語の置換・同義語の挿入・語の入れ替え・語の削除のどれか 1 つを、文の約 1 割の語に施す
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)   # 日本語 WordNet (NICT) を含む多言語 WordNet
from fugashi import Tagger
from nltk.corpus import wordnet as wn

tagger = Tagger()

def synonyms(word):
    return sorted({l.replace("_", "") for s in wn.synsets(word, lang="jpn") for l in s.lemma_names("jpn")} - {word})

def eda_once(title, rng, alpha):
    tokens = list(tagger(title))
    words = [w.surface for w in tokens]
    nouns = [i for i, w in enumerate(tokens) if w.feature.pos1 == "名詞" and w.feature.pos2 != "固有名詞" and synonyms(w.surface)]
    n = max(1, round(alpha * len(words)))
    op = rng.choice(["置換", "挿入", "入替", "削除"])
    w = words[:]
    if op == "置換":       # 名詞を WordNet の同義語に置き換える（活用する語は形が崩れるので名詞だけ）
        for i in rng.sample(nouns, min(n, len(nouns))):
            w[i] = rng.choice(synonyms(w[i]))
    elif op == "挿入":     # どれかの名詞の同義語を、ランダムな位置に入れる
        for _ in range(n if nouns else 0):
            w.insert(rng.randrange(len(w) + 1), rng.choice(synonyms(words[rng.choice(nouns)])))
    elif op == "入替":     # ランダムな 2 語を入れ替える
        for _ in range(n):
            i, j = rng.sample(range(len(w)), 2)
            w[i], w[j] = w[j], w[i]
    else:                  # 各語を確率 alpha で消す
        w = [x for x in w if rng.random() > alpha] or [rng.choice(w)]
    return "".join(w), op

def eda(title, rng, alpha=0.1, tries=10):
    for _ in range(tries):        # 何も消えなかった・置き換える名詞がなかったなどで元の文と同じになったら、やり直す
        new, op = eda_once(title, rng, alpha)
        if new != title:
            break
    return new, op

rng = random.Random(0)
eda_out = {i: [eda(pool.title[i], rng) for _ in range(2)] for i in used}
augs["EDA"] = {i: [t for t, _ in v] for i, v in eda_out.items()}
for i in used[:6]:
    print(pool.title[i], "\n   →", " / ".join(f"[{op}] {t}" for t, op in eda_out[i]))

みのもんた引退で変わる芸能界の勢力図 
   → [挿入] みのもんた引退で退任変わる芸能界の勢力図 / [入替] みのもんた勢力で変わる芸能界の引退図
ピザの起源は“韓国”とするCMが話題に 
   → [入替] ピザ話題起源は“韓国”とするCMがのに / [置換] ピザの起源は“韓国”とする糎が話題に
ビッグダディに「実はけっこう金持ち」疑惑が浮上 
   → [置換] ビッグダディに「実はけっこう富豪」疑惑が浮上 / [入替] ビッグダディに「実はけっこう金持ち」が疑惑浮上
当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々の大騒ぎ 
   → [挿入] 当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々のイグゼクティヴ大騒ぎ大物 / [置換] 其節18歳の沢尻エリカ、暴力団幹部とＶＩＰ席で堂々の大騒ぎ
和田アキ子が秋葉原で激怒！　「笑っていいとも」で告白 
   → [置換] 和田アキ子が秋葉原で憤激！　「笑っていいとも」で承認 / [置換] 和田アキ子が秋葉原で憤怒！　「笑っていいとも」で御了承
釈由美子のありえない勘違いに仰天 
   → [削除] 由美子のありえない勘違いに仰天 / [挿入] 釈由美子のありえない勘違いに仰天驚き


In [4]:
# (2) 逆翻訳: 日本語 → 英語 → 日本語と往復させる（機械翻訳の章の「逆翻訳で擬似対訳データを作る」と同じ考え方）
# Colab の T4 で 3 分程度
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 は bf16 が使えないので float16
DTYPE = torch.bfloat16 if BF16 else torch.float16
name = "facebook/nllb-200-distilled-600M"              # Meta の NLLB（CC BY-NC 4.0、非商用）
mt_tok = AutoTokenizer.from_pretrained(name)
mt = AutoModelForSeq2SeqLM.from_pretrained(name, dtype=DTYPE).to("cuda").eval()

@torch.no_grad()
def translate(texts, src, tgt, n=1, num_beams=4, batch_size=64):
    mt_tok.src_lang = src
    out = []
    for i in range(0, len(texts), batch_size):
        enc = mt_tok(texts[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=128).to("cuda")
        gen = mt.generate(**enc, forced_bos_token_id=mt_tok.convert_tokens_to_ids(tgt), num_beams=num_beams,
                          num_return_sequences=n, max_length=128)
        out += mt_tok.batch_decode(gen, skip_special_tokens=True)
    return out

titles = [pool.title[i] for i in used]
en = translate(titles, "jpn_Jpan", "eng_Latn", num_beams=1)   # 行きは貪欲法で軽くする
ja = translate(en, "eng_Latn", "jpn_Jpan", n=2)          # ビームの上位 2 つを言い換えにする
augs["逆翻訳"] = {i: ja[2 * k:2 * k + 2] for k, i in enumerate(used)}
for k in range(6):
    print(titles[k], "\n   →", en[k], "\n   →", " / ".join(augs["逆翻訳"][used[k]]))
del mt; torch.cuda.empty_cache()

みのもんた引退で変わる芸能界の勢力図 
   → The power of the entertainment industry that changes after retirement. 
   → 退職後に変化するエンターテインメント業界の力 / 退職後に変化するエンターテインメント産業の力
ピザの起源は“韓国”とするCMが話題に 
   → The origins of pizza are a matter of debate. 
   → ピザの起源は議論の余地がある. / ピザの起源は議論の余地があります.
ビッグダディに「実はけっこう金持ち」疑惑が浮上 
   → Big Daddy's suspicion is that he's really rich. 
   → ビッグパパの疑いは,彼が本当に金持ちだということです. / ビッグパパの疑いは,彼は本当に金持ちだということです.
当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々の大騒ぎ 
   → And then there was the 18-year-old Zsuzsa Erica, the head of the gang, and the VIP seat. 
   → それから18歳のツーサ・エリカもいました ギャングのリーダーと VIP席です / それから18歳のツーサ・エリカ ギャングのリーダー そしてVIP席もありました
和田アキ子が秋葉原で激怒！　「笑っていいとも」で告白 
   → Akiko was furious at the Akita Plain. 
   → アキコはアキタ平原に怒っていた. / アキコはアキタ平原に対して怒っていた.
釈由美子のありえない勘違いに仰天 
   → I'm so excited about the unlikely differences. 
   → 不可能な違いに とても興奮しています / 不可能な違いに興奮しています


In [5]:
# (3) LLM による言い換え: 指示チューニング済みの 3B モデルに見出しを書き換えさせる
# Colab の T4 で 2 分程度
import sentencepiece as spm
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"    # SB Intuitions（MIT ライセンス）
tok = AutoTokenizer.from_pretrained(name)
print("差し替える前:", tok.tokenize("みのもんた引退、新卒初任給"))
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tok.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
print("差し替えた後:", tok.tokenize("みのもんた引退、新卒初任給"))
tok.padding_side = "left"
lm = AutoModelForCausalLM.from_pretrained(name, dtype=DTYPE).to("cuda").eval()
print(f"GPU メモリ: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

def prompt(title):
    msg = f"次のニュース記事の見出しを、意味を変えずに別の言い方で書き換えてください。書き換えた見出しだけを 1 行で出力してください。\n\n見出し: {title}"
    return tok.apply_chat_template([{"role": "user", "content": msg}], add_generation_prompt=True, tokenize=False)

def clean(s):
    s = s.strip().split("\n")[0].strip()
    return s[1:-1] if s.startswith("「") and s.endswith("」") else s

@torch.no_grad()
def paraphrase(texts, n=2, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok([prompt(t) for t in texts[i:i + batch_size]], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = lm.generate(**enc, max_new_tokens=40, do_sample=True, temperature=0.8, top_p=0.9, num_return_sequences=n,
                          pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id)
        out += [clean(s) for s in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return out

torch.manual_seed(0)
para = paraphrase(titles)
augs["LLM"] = {i: para[2 * k:2 * k + 2] for k, i in enumerate(used)}
for k in range(6):
    print(titles[k], "\n   →", " / ".join(augs["LLM"][used[k]]))
del lm; torch.cuda.empty_cache()

差し替える前: ['み', 'のも', 'ん', 'た', '引退', '、', '新卒', '初', '任', '給']
差し替えた後: ['みの', 'もん', 'た', '引退', '、', '新卒', '初任給']
GPU メモリ: 6.7 GB
みのもんた引退で変わる芸能界の勢力図 
   → みのもんた氏の引退が占う芸能界の新たな構図 / みのもんた氏の引退が引き起こす芸能界の勢力変動
ピザの起源は“韓国”とするCMが話題に 
   → 韓国で放送されているピザ起源CMが海外で物議を醸す / ピザの起源を主張した韓国企業CMが物議を醸す
ビッグダディに「実はけっこう金持ち」疑惑が浮上 
   → ビッグダディに新たな経済背景の憶測が浮上: 実は quite the financier なのでは？ / ビッグダディの経済的な側面に注目集まる: 意外な富豪説が浮上
当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々の大騒ぎ 
   → 沢尻エリカ、18歳の時に暴力団幹部とVIP席で目撃された騒動の様子 / 沢尻エリカ、18歳で暴力団幹部と高級席での派手な行動が明るみに。
和田アキ子が秋葉原で激怒！　「笑っていいとも」で告白 
   → 和田アキ子、秋葉原でのIncidentを「笑っていいとも」で語る / 和田アキ子、秋葉原での経験を「笑っていいとも」で激白：激怒の体験を語る
釈由美子のありえない勘違いに仰天 
   → 釈由美子、勘違いに関する驚きのニュース浮上 / 釈由美子の勘違い報道に衝撃走る


In [6]:
# 拡張なし・各拡張・本物 3 倍で学習し、同じテストデータで正解率を比べる（分類器は TF-IDF + LR と、文埋め込み + LR）
# 5 通り × 5 条件 × 2 分類器 = 50 回の学習を CPU で行うので、Colab で 3 分程度
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

st = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
all_texts = sorted(set(test["title"]) | set(pool["title"]) | {t for a in augs.values() for v in a.values() for t in v})
vec = dict(zip(all_texts, st.encode(["query: " + t for t in all_texts], normalize_embeddings=True, batch_size=256)))
embed = lambda texts: np.stack([vec[t] for t in texts])
X_test = embed(test["title"])

def evaluate(texts, labels):
    tfidf = make_pipeline(TfidfVectorizer(analyzer="char", ngram_range=(1, 3), sublinear_tf=True), LogisticRegression(C=10, max_iter=1000))
    acc1 = tfidf.fit(texts, labels).score(test["title"], test["label"])
    acc2 = LogisticRegression(C=10, max_iter=1000).fit(embed(texts), labels).score(X_test, test["label"])
    return acc1, acc2

res = {}
for s in SEEDS:
    d = small[s]
    res["拡張なし", s] = evaluate(list(d["title"]), list(d["label"]))
    for method, a in augs.items():   # 元の 90 件 + 言い換え 180 件
        res[method, s] = evaluate(list(d["title"]) + [t for i in d.index for t in a[i]],
                                  list(d["label"]) + [l for l in d["label"] for _ in range(2)])
    res["本物 3 倍", s] = evaluate(list(real3x[s]["title"]), list(real3x[s]["label"]))

rows = []
for method in ["拡張なし", *augs, "本物 3 倍"]:
    acc, base = np.array([res[method, s] for s in SEEDS]), np.array([res["拡張なし", s] for s in SEEDS])
    row = {"手法": method}
    for j, clf_name in enumerate(["TF-IDF", "埋め込み"]):
        diff = acc[:, j] - base[:, j]                                   # 同じ種どうしの差（対応のある比較）
        half = stats.t.ppf(0.975, len(diff) - 1) * diff.std(ddof=1) / np.sqrt(len(diff))
        row[f"{clf_name} 正解率"] = f"{acc[:, j].mean():.3f} ± {acc[:, j].std(ddof=1):.3f}"
        row[f"{clf_name} 差 [95%CI]"] = f"{diff.mean():+.3f} [{diff.mean() - half:+.3f}, {diff.mean() + half:+.3f}]"
    rows.append(row)
pd.set_option("display.width", 200)
print(pd.DataFrame(rows).to_string(index=False))

    手法    TF-IDF 正解率        TF-IDF 差 [95%CI]      埋め込み 正解率          埋め込み 差 [95%CI]
  拡張なし 0.606 ± 0.014 +0.000 [+0.000, +0.000] 0.650 ± 0.005 +0.000 [+0.000, +0.000]
   EDA 0.604 ± 0.017 -0.002 [-0.009, +0.005] 0.644 ± 0.007 -0.006 [-0.012, -0.000]
   逆翻訳 0.573 ± 0.010 -0.033 [-0.047, -0.018] 0.599 ± 0.011 -0.051 [-0.069, -0.034]
   LLM 0.601 ± 0.010 -0.004 [-0.020, +0.011] 0.639 ± 0.006 -0.011 [-0.020, -0.002]
本物 3 倍 0.696 ± 0.014 +0.091 [+0.057, +0.124] 0.715 ± 0.005 +0.065 [+0.056, +0.074]


In [7]:
# 拡張でラベルが変わっていないか: 学習用プールのうち言い換えの対象でない文（約 5500 件）で学習した分類器を「審判」にして、
# 元の文と言い換えた文の予測ラベルを調べる。元の文を審判の学習に入れると、答えを覚えているので比較にならない
# （この審判は分析のためだけのもの。実際に少量のデータしかない場面では使えない）
rest = pool.drop(used)
judge = LogisticRegression(C=10, max_iter=1000).fit(embed(rest["title"]), rest["label"])
gold = np.array([pool.label[i] for i in used])
pred_orig = judge.predict(embed(titles))
print(f"元の文     : 審判の予測が元のラベルと一致 {np.mean(pred_orig == gold):.1%}")
for method, a in augs.items():
    aug_texts = [t for i in used for t in a[i]]
    pred = judge.predict(embed(aug_texts))
    cos = (embed(aug_texts) * np.repeat(embed(titles), 2, axis=0)).sum(axis=1)
    print(f"{method:6}: 審判の予測が元のラベルと一致 {np.mean(pred == np.repeat(gold, 2)):.1%} / 元の文との埋め込みの類似度 (平均) {cos.mean():.3f}")

# 元の文は正しく分類されるのに、言い換えると別のクラスになる例
for method, a in augs.items():
    print(f"\n--- {method}")
    shown = 0
    for k, i in enumerate(used):
        p = judge.predict(embed(a[i][:1]))[0]
        if pred_orig[k] == gold[k] and p != gold[k]:
            print(f"[{gold[k]} → {p}] {titles[k]}\n    → {a[i][0]}")
            shown += 1
        if shown == 4:
            break

元の文     : 審判の予測が元のラベルと一致 83.2%
EDA   : 審判の予測が元のラベルと一致 79.4% / 元の文との埋め込みの類似度 (平均) 0.982
逆翻訳   : 審判の予測が元のラベルと一致 53.2% / 元の文との埋め込みの類似度 (平均) 0.900
LLM   : 審判の予測が元のラベルと一致 71.3% / 元の文との埋め込みの類似度 (平均) 0.954

--- EDA
[topic-news → movie-enter] 当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々の大騒ぎ
    → 当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々のイグゼクティヴ大騒ぎ大物
[topic-news → movie-enter] チャン・グンソク韓国で嫌われすぎて映画大コケか？
    → チャン・グンソク韓国で嫌われすぎて映画大コケかエクラン？
[topic-news → sports-watch] 西岡剛内野手とモデルの徳澤直子が離婚、原因は西岡の不倫か
    → 剛内野手とモデル徳澤直子が離婚、西岡のか
[topic-news → dokujo-tsushin] 「“彼女がいなくてもいい”は一種の病気」恋愛学の権威の発言が話題
    → 「元老“彼女がいなくてもいい”は一種の病気」物病恋愛学の権威の発言が話題

--- 逆翻訳
[topic-news → dokujo-tsushin] みのもんた引退で変わる芸能界の勢力図
    → 退職後に変化するエンターテインメント業界の力
[topic-news → dokujo-tsushin] ピザの起源は“韓国”とするCMが話題に
    → ピザの起源は議論の余地がある.
[topic-news → movie-enter] 当時18歳の沢尻エリカ、暴力団幹部とVIP席で堂々の大騒ぎ
    → それから18歳のツーサ・エリカもいました ギャングのリーダーと VIP席です
[topic-news → dokujo-tsushin] 釈由美子のありえない勘違いに仰天
    → 不可能な違いに とても興奮しています

--- LLM
[topic-news → it-life-hack] ビッグダディに「実はけっこう金持ち」疑惑が浮上
    → ビッグダディに新たな経済背

- EDA は Wei and Zou (EMNLP-IJCNLP 2019) が提案した 4 つの操作（同義語の置換・挿入、語の入れ替え・削除）。英語では同義語を WordNet から取る。日本語では分かち書きしてから、日本語 WordNet の同義語を使う。ここでは活用で形が崩れないように、同義語を使う操作は名詞だけにした。それでも「CM」が長さの単位（センチメートル）を表す「糎」に置き換わるなど、語義の取り違えは避けられない。
- 逆翻訳はもともと機械翻訳の学習データを増やす方法（Sennrich ら, ACL 2016。機械翻訳の章の「逆翻訳で擬似対訳データを作る」）。分類では、往復させた文を元の文の言い換えとして使う（Xie らの UDA, NeurIPS 2020 など）。翻訳では人が書いた文が訳文側に残るが、ここでは往復した機械訳そのものを学習に使うので、誤訳がそのまま混ざる。
- この実験（タイトルで 9 クラス、各クラス 10 件、5 通りの学習データ）では、どの拡張も正解率を上げなかった。EDA と LLM の言い換えは、差の 95% 信頼区間が 0 をまたぐか 0 以下（平均で最大 1.1 ポイント減）で、逆翻訳は 3〜5 ポイント下がった。いっぽう、本物のデータを 3 倍にすると 6.5〜9 ポイント上がる。1 件を言い換えた 2 件は、本物の 2 件の代わりにならない（データを増やす効果はこの章の「学習データを増やすと精度がどう変わるか調べる（学習曲線）」も参照）。
- 差が小さいときは 1 回の結果で判断せず、同じ学習データ（同じ種）で拡張の有無を比べ、その差の平均と信頼区間を見る（対応のある比較）。この実験でも、種による正解率の標準偏差（TF-IDF で 0.01〜0.02）は、EDA や LLM の言い換えによる差と同じくらいか、それより大きい。なお、この信頼区間は学習データの選び方のばらつきだけを表す。テストデータと言い換え（EDA の乱数、LLM のサンプリング）は 1 回しか作っていない。
- 審判の分類器（言い換えの対象でない文で学習）で見ると、元の文の 83% が元のラベルに分類されるのに対し、EDA は 79%、LLM は 71%、逆翻訳は 53% だった。往復翻訳では人名が落ちたり別人（「ツーサ・エリカ」）になったりして、見出しの意味が壊れている。LLM の言い換えは意味がおおむね保たれていても、話題の重心が移って（「映画大コケか」→「主演の韓国映画、予想外の低成績」）別のカテゴリらしくなることがある。
- Longpre ら (Findings of EMNLP 2020) は、EDA と逆翻訳が、ファインチューニングした事前学習済み Transformer の精度を安定して上げないと報告している。ここでは埋め込みモデルを固定してロジスティック回帰を学習しており、設定は違う。この実験に限っていえば、埋め込みを使った分類器のほうが、拡張による低下が大きかった。
- transformers 5 系では、sarashina2.2 のトークナイザーが本来の SentencePiece (Unigram) と違う分割になる（上の出力では「みのもんた」が「み / のも / ん / た」、「初任給」が 1 字ずつに分かれる）ので、上のようにモデル部分を Unigram に戻して使う。直したあとも、言い換えに英語が混ざる（「quite the financier」「Incident」）ことがある。言い換えが元の文から離れすぎていないか（埋め込みの類似度）や、ラベルが変わっていないかを確かめてから使う。機械翻訳の章の「擬似対訳データを往復翻訳の一致度でふるいにかける」と同じ発想で、類似度の低いものを捨てる方法もある（ここでは試していない）。
- データは CC BY-ND（改変禁止）なので、言い換えたデータを公開・配布してはいけない。NLLB は CC BY-NC 4.0（非商用）、sarashina2.2 は MIT、multilingual-e5-small は MIT、日本語 WordNet は NICT の独自ライセンス（著作権表示を残せば利用・改変・再配布可）。